# Hult Form — Using `general_llm_app`

This notebook shows how to use the simplified `LLM` class to generate form questions from KPI definitions.

Compare this to `hult_form.ipynb` — same result, way less boilerplate.

In [47]:
import json
from pandas._typing import F
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

from pathlib import Path
import sys
sys.path.append(str(Path.cwd().parent))
from llm import LLM

llm = LLM(model="gpt-4o-mini")

In [48]:
FOLDER_PATH = "tasks/hult_evaluation/impact_measurement"             
DATA_PATH   = "../data/app.json"          

In [49]:
# Load task config
task_config = json.loads((Path("..") / FOLDER_PATH / "task.json").read_text())
var_map     = task_config["variables"]            
meta_cols   = task_config.get("metadata_columns", [])

In [50]:
df = pd.read_json("../data/app.json")

In [51]:
df["pitch_transcript"] = """
        Dreamland Ghana. Now, Dreamland Coco
Ghana is company that is addressing a
longlasting issue which has long existed
in Ghana. Throughout the years, Ghana
has been one of the top producers of
cocoa in the world. We are the second
largest producer of cocoa in the world
and we also have large amounts of cashew
nuts and granut nuts that we produce
locally here. The problem, however, is
even though the global chocolate
industry is worth over 146 billion US,
farmers in Ghana only attain a little
over2 billion US annually. This is a
problem. This this is an economic
problem and countries and systems run
with money. Now, how does Greenland Poco
Poco plans to solve this problem? We
plan to solve this problem with our
vertically included
company. What that does what does that
mean? What that means is that Dreamland
Cocoa has access to the farms. It has
access to the cocoa. It it has access to
sourcing. It has access to all these
because the founders of the company are
all from cocoa far cocoa producing
families. Now, how do we plan to use our
value position and our engineering
expertise to solve a problem in Ghana?
We are building a company from the
ground up which is vertically included.
All farmers on deck. Farmers get a share
in the company. They get to contribute
in the building of the company from the
ground up. Now, what does that mean? It
means that we have access to our
sourcing material. We have access to
cashew. We have access to granite. We
have access to cocoa. We have access to
all these. We have the engineering
expertise. We have the materials to
produce chocolate to the global market.
What is this going to do for the
farmers? This is going to increase their
economic value of the chocolate or of
the cocoa that they produce. is going to
provide economic value for cashew which
is also primarily exported out from
Africa to Asia then exported from Asia
to the global market for sales. Large
sum of the economic value of all these
products are exploited by large
conglomerates which are not in
indigenous African companies.
Now, Dreamland Coco is working on all of
these of the problem simultaneously
whilst impacting the community. We are
impacting the community through job
creation. We are impacting the community
through more value addition and more
value is to be captured for the local
farmers. We are impacting the community
through
ethically sourced cocoa. That is cocoa
that has not been sourced through
inhumane human labor or even child
labor. Another point that Dreamland is
working towards on is our cashew
industry in Ghana. is largely exported
as raw beans countries and then
processed to be shipped to the US and UK
markets for large marketable value. Now,
Dreamland is stepping in that gap.
Dreamland is taking the cashew which is
produced locally here in Africa,
processing it with the cocoa into
beautiful
chocolate coated cashew nuts and other
products and selling them to the global
market. Now how does Halt Pry comes into
our picture? With our traction as
dreamland horse price coming in is going
to give us the money to be able to touch
more lives to impact more businesses
locally and to produce more value for
the farmers who work so hard to produce
these valuable products they need. Thank
you so much and I hope you consider our
health price application.
    """

In [52]:
columns = ['Which SDG 1 Target(s) does your solution align with? (Select up to 2)', 'Which SDG 8 Target(s) does your solution align with? (Select up to 2)',
       'pitch_transcript']

In [53]:
df["sdg_targets"] = (
    df[columns[0]].astype(str)
    + "\n"
    + df[columns[1]].astype(str)
)

In [54]:
def make_variables(row):
    return {placeholder: str(row[col]) for placeholder, col in var_map.items()}

In [55]:
results = llm.batch(
    f"{FOLDER_PATH}/prompt.md",
    rows=df.to_dict("records"),
    variables_fn=make_variables,
    schema_path=f"{FOLDER_PATH}/schema.json",
)

for r, (_, row) in zip(results, df.iterrows()):
    for col in meta_cols:
        r[col.lower().replace(" ", "_")] = row[col]

  [1/1] row 1

✅ Done — 1 results.


In [56]:
# Preview first result
print(json.dumps(results[0], indent=2))

{
  "sdg_alignment": {
    "is_aligned": true,
    "sdgs_identified": [
      "SDG 1",
      "SDG 8"
    ],
    "reasoning": "The venture addresses poverty eradication and economic growth by providing farmers with a share in the company and increasing the value of their products."
  },
  "kpi_assessment": {
    "has_clear_kpis": false,
    "kpis_identified": [],
    "reasoning": "While the venture discusses community impact and economic benefits, it lacks specific, measurable KPIs to quantify its impact."
  },
  "revenue_impact_link": {
    "impact_grows_with_revenue": "potential",
    "reasoning": "The business model suggests that as revenue increases from selling value-added products, the economic benefits to farmers could also increase, but this link is not explicitly defined."
  },
  "score": 3,
  "score_justification": "The venture is aligned with relevant SDGs and has a potential for impact growth, but it lacks clear KPIs for measuring that impact."
}
